In [1]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import pandas as pd
import json
import time

print("modules imported succesfully")

/Users/chaitanyayadav/personal/code/ai_projects/hf_models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modules imported succesfully


In [2]:
class StudentProfile(BaseModel):
    degree_level: str = Field(description="student's current degree level")
    percentage: float = Field(description="student's latest percentage in the degree")
    is_reserved: bool = Field(desctiption="whether student belong to a reserved category or not")
    academic_background: str = Field(description="student's degree name and subject")
    subjects: List[str] = Field(description="subjects studied in the course")
    interests: List[str] = Field(description="student's area of interests")
    career_goal: str = Field(desctiption="student's career goal")
    preferred_skills: List[str] = Field(description="student's preferred skills to learn")
    preferred_domain: str = Field(description="student's preferred domain")

class StudentData(BaseModel):
    students: List[StudentProfile] = Field(description="A list containing the data of the students")
print("class initialised successfully")


class initialised successfully


/var/folders/0n/3157pzwj13xd5kvdtpbqgv440000gn/T/ipykernel_67614/4045606446.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desctiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  is_reserved: bool = Field(desctiption="whether student belong to a reserved category or not")
/var/folders/0n/3157pzwj13xd5kvdtpbqgv440000gn/T/ipykernel_67614/4045606446.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desctiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  career_goal: str = Field(desctiption="student's career goal")


In [12]:
class EligibilityStruct(BaseModel):
    required_subjects: List[str]
    allowed_subjects: List[str]
    min_subject_match_count: Optional[int]
    preferred_domains: List[str]
    is_open_degree: bool

In [ ]:
system_prompt = """
You are a keyword extraction system for eligibility matching.

Your task is to convert a university eligibility sentence into a dense keyword sentence optimized for similarity matching.

-------------------------------------

RULES:

1. DO NOT explain anything
2. DO NOT generate full sentences
3. DO NOT include filler words
4. DO NOT include stopwords (like "and", "or", "with", "any")
5. DO NOT include words like location names, numbers or symbols
5. DO NOT repeat words
6. DO NOT hallucinate or add information not present
7. Output MUST be a single line of space-separated keywords

-------------------------------------

WHAT TO EXTRACT:

Focus ONLY on meaningful eligibility signals:

1. Degree types:
   Examples: BSc, BTech, BA, MSc, MTech, BCom

2. Core subjects:
   Examples: Botany, Biology, Chemistry, Physics, Mathematics

3. Related subjects (if mentioned)

4. Domain-level meaning:
   Examples: LifeSciences, Engineering, Commerce, Arts

-------------------------------------

NORMALIZATION RULES:

- "Bachelor of Science" → BSc
- "B.Sc." → BSc
- "Computer Science" → ComputerScience
- Multi-word subjects → combine words (no spaces)

-------------------------------------

EXAMPLES:

Input:
"B.Sc. in Biotechnology or related fields"

Output:
BSc Biotechnology LifeSciences

Input:
"Bachelor degree in any discipline"

Output:
Bachelor AnyDomain

Input:
"B.Sc. with Botany and any two of the following: Chemistry, Zoology, Microbiology"

Output:
BSc Botany Chemistry Zoology Microbiology LifeSciences

-------------------------------------

Now extract keywords from the given eligibility text.
"""

In [3]:
import os
key = os.getenv("GOOGLE_API_KEY","")
gen_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, api_key=key)
local_model = ChatOllama(model="qwen2.5:1.5b", temperature=0)

df = pd.read_csv("../recommender_system/final_data3.csv")

results = []

for i, row in df.iterrows():
    
    try:
        conversation = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=str(row['eligibility']))
        ]

        response = local_model.invoke(conversation)
        result = {
            "eligibility_keywords": response.content.lower().split(" ")
        }
        results.append(result)
        print(f"Processed row {i+1}")

    except Exception as e:
        print(f"Error generating the data on row {i+1}: {e}")
        results.append({
            "eligibility_keywords": "[]"
        })
    
eligibility_df = pd.DataFrame(results)

final_df = pd.concat([df, eligibility_df], axis=1)
final_df.to_csv("courses_data.csv", index=False)
print("Data generation complete")
        

Processed row 1
Processed row 2
Processed row 3
Processed row 4
Processed row 5
Processed row 6
Processed row 7
Processed row 8
Processed row 9
Processed row 10
Processed row 11
Processed row 12
Processed row 13
Processed row 14
Processed row 15
Processed row 16
Processed row 17
Processed row 18
Processed row 19
Processed row 20
Processed row 21
Processed row 22
Processed row 23
Processed row 24
Processed row 25
Processed row 26
Processed row 27
Processed row 28
Processed row 29
Processed row 30
Processed row 31
Processed row 32
Processed row 33
Processed row 34
Processed row 35
Processed row 36
Processed row 37
Processed row 38
Processed row 39
Processed row 40
Processed row 41
Processed row 42
Processed row 43
Processed row 44
Processed row 45
Processed row 46
Processed row 47
Processed row 48
Processed row 49
Processed row 50
Processed row 51
Processed row 52
Processed row 53
Processed row 54
Processed row 55
Processed row 56
Processed row 57
Processed row 58
Processed row 59
Proces

In [7]:
import json

file = open("students_data.json", "r")
contents = json.load(file)

df = pd.DataFrame(contents)
df.to_csv("students_data.csv")